In [1]:
import os

SRC_DIR = "/kaggle/input/datasets/swetha344/pyg-210-cu128-wheels"
DST_DIR = "/kaggle/working/fixed_wheels"

os.makedirs(DST_DIR, exist_ok=True)

rename_map = {
    "torch_scatter-2.1.2pt210cu128-cp312-cp312-linux_x86_64.whl": "torch_scatter-2.1.2+pt210cu128-cp312-cp312-linux_x86_64.whl",
    "torch_cluster-1.6.3pt210cu128-cp312-cp312-linux_x86_64.whl": "torch_cluster-1.6.3+pt210cu128-cp312-cp312-linux_x86_64.whl",
    "pyg_lib-0.4.0pt210cu128-cp312-cp312-linux_x86_64.whl": "pyg_lib-0.4.0+pt210cu128-cp312-cp312-linux_x86_64.whl",
    "torch_geometric-2.7.0-py3-none-any.whl": "torch_geometric-2.7.0-py3-none-any.whl",
    "ogb-1.3.6-py3-none-any.whl": "ogb-1.3.6-py3-none-any.whl"
}

for old_name, new_name in rename_map.items():
    src = os.path.join(SRC_DIR, old_name)
    dst = os.path.join(DST_DIR, new_name)
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)

!pip install --no-index --no-deps --force-reinstall /kaggle/working/fixed_wheels/*.whl

Processing ./fixed_wheels/ogb-1.3.6-py3-none-any.whl
Processing ./fixed_wheels/torch_cluster-1.6.3+pt210cu128-cp312-cp312-linux_x86_64.whl
Processing ./fixed_wheels/torch_geometric-2.7.0-py3-none-any.whl
Processing ./fixed_wheels/torch_scatter-2.1.2+pt210cu128-cp312-cp312-linux_x86_64.whl


In [2]:
import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

In [3]:
import os
import pandas as pd
import torch
from ogb.nodeproppred import PygNodePropPredDataset

# 1. Load the graph
dataset = PygNodePropPredDataset(name='ogbn-arxiv')
data = dataset[0]

# 2. Load Mapping Files
# The 'nodeidx2paperid' file maps: Node Index -> MAG Paper ID
path = os.path.join(dataset.root, 'mapping', 'nodeidx2paperid.csv.gz')
nodeidx2paperid = pd.read_csv(path)

# The 'titleabs' file contains: MAG Paper ID -> Title/Abstract
if not os.path.exists('titleabs.tsv'):
    !wget https://snap.stanford.edu/ogb/data/misc/ogbn_arxiv/titleabs.tsv.gz
    !gunzip titleabs.tsv.gz

titleabs = pd.read_csv('titleabs.tsv', sep='\t', names=['paper id', 'title', 'abstract'])

nodeidx2paperid['paper id'] = pd.to_numeric(nodeidx2paperid['paper id'], errors='coerce')
titleabs['paper id'] = pd.to_numeric(titleabs['paper id'], errors='coerce')
nodeidx2paperid = nodeidx2paperid.dropna(subset=['paper id']).astype({'paper id': 'int64'})
titleabs = titleabs.dropna(subset=['paper id']).astype({'paper id': 'int64'})

print("Aligning text to node indices...")
# We merge on nodeidx2paperid so the final dataframe order matches the Graph Node Indices (0 to 169,342)
df = pd.merge(nodeidx2paperid, titleabs, on='paper id', how='left')

# Handle missing text (if any papers in the graph aren't in the tsv)
df['title'] = df['title'].fillna('Unknown Title')
df['abstract'] = df['abstract'].fillna('Unknown Abstract')
df['text'] = df['title'] + ". " + df['abstract']

texts = df['text'].tolist()
print(f"Successfully aligned {len(texts)} texts.")

Downloaded 0.08 GB: 100%|██████████| 81/81 [00:14<00:00,  5.44it/s]
Processing...


Extracting dataset/arxiv.zip
Loading necessary files...
This might take a while.
Processing graphs...


100%|██████████| 1/1 [00:00<00:00, 9731.56it/s]


Converting graphs into PyG objects...


100%|██████████| 1/1 [00:00<00:00, 392.54it/s]

Saving...



Done!
/usr/local/lib/python3.12/dist-packages/ogb/nodeproppred/dataset_pyg.py:69: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  self.data, self.slices = torch.load(self.processed_paths[0])


--2026-04-22 15:49:01--  https://snap.stanford.edu/ogb/data/misc/ogbn_arxiv/titleabs.tsv.gz
Resolving snap.stanford.edu (snap.stanford.edu)... 171.64.75.80
Connecting to snap.stanford.edu (snap.stanford.edu)|171.64.75.80|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 70213527 (67M) [application/x-gzip]
Saving to: ‘titleabs.tsv.gz’

titleabs.tsv.gz     100%[===================>]  66.96M  19.6MB/s    in 3.8s    

2026-04-22 15:49:05 (17.7 MB/s) - ‘titleabs.tsv.gz’ saved [70213527/70213527]

Aligning text to node indices...
Successfully aligned 169343 texts.


In [4]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Prepare Graph
data.edge_index = to_undirected(data.edge_index)
data.x = torch.load('/kaggle/input/datasets/swetha344/ogbn-arxiv-roberta-embeddings/arxiv_roberta_features.pt').to(torch.float)
data = data.to(device)

split_idx = dataset.get_idx_split()
train_idx = split_idx['train'].to(device)
valid_idx = split_idx['valid'].to(device)
test_idx = split_idx['test'].to(device)

/tmp/ipykernel_55/3115342646.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  data.x = torch.load('/kaggle/input/datasets/swetha344/ogbn-arxiv-roberta-embeddings/arxiv_roberta_features.pt').to(torch.float)


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv

# Reversible Block Implementation
class RevGATBlock(nn.Module):
    def __init__(self, in_channels, out_channels, heads, dropout):
        super().__init__()
        self.norm = nn.BatchNorm1d(in_channels)
        self.conv = GATConv(in_channels, out_channels // heads, heads=heads, dropout=dropout)
        
    def forward(self, x, edge_index):
        out = self.norm(x)
        out = F.relu(out)
        out = self.conv(out, edge_index)
        return out

class RevGAT(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers, heads, dropout):
        super().__init__()
        # Initial projection to hidden space
        self.lin_in = nn.Linear(in_channels, hidden_channels)
        
        self.layers = nn.ModuleList()
        for _ in range(num_layers):
            self.layers.append(RevGATBlock(hidden_channels // 2, hidden_channels // 2, heads, dropout))
            
        self.lin_out = nn.Linear(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.lin_in(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        
        # Split features for reversible path
        x1, x2 = torch.chunk(x, 2, dim=-1)
        
        for layer in self.layers:
            # Standard Reversible Update:
            # x1 = x1 + f(x2)
            # Swap(x1, x2)
            new_x1 = x1 + layer(x2, edge_index)
            x1, x2 = x2, new_x1
            
        x = torch.cat([x1, x2], dim=-1)
        x = F.relu(x)
        return self.lin_out(x).log_softmax(dim=-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion = torch.nn.CrossEntropyLoss()

In [6]:
import torch
import numpy as np
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Configuration
num_runs = 10
epochs = 500
eval_steps = 10 
all_test_results = []

global_best_test_acc = 0.0
global_save_path = 'overall_best_revgat_model.pt'

# Using Label Smoothing to bridge the Val/Test gap
criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.1)

for run in range(num_runs):
    seed = run 
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    
    print(f"\n>>> Starting Run {run+1}/{num_runs} (Seed: {seed})")
    
    # 1. Model init with RevGAT architecture
    # Note: in_channels=768 for RoBERTa
    model = RevGAT(
        in_channels=768,
        hidden_channels=256, 
        out_channels=dataset.num_classes,
        num_layers=3,
        heads=4,
        dropout=0.5
    ).to(device)

    # 2. Optimizer and Scheduler
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
    
    # mode='max' because we want to maximize validation accuracy
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10, min_lr=1e-5)
    
    best_run_val_acc = 0
    best_run_test_acc = 0

    # 3. Training Loop
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = criterion(out[train_idx], data.y[train_idx].squeeze())
        loss.backward()
        
        # Adding Gradient Clipping for RevGAT stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()

        # Check performance more frequently
        if epoch % eval_steps == 0:
            model.eval()
            with torch.no_grad():
                logits = model(data.x, data.edge_index)
                y_pred = logits.argmax(dim=-1, keepdim=True)
                
                val_acc = (y_pred[valid_idx] == data.y[valid_idx]).sum().item() / valid_idx.size(0)
                test_acc = (y_pred[test_idx] == data.y[test_idx]).sum().item() / test_idx.size(0)

                # Step the scheduler based on validation accuracy
                scheduler.step(val_acc)
                
                # Log current learning rate
                current_lr = optimizer.param_groups[0]['lr']

                if epoch % 10 == 0:
                    print(f"Epoch {epoch:03d}: Val: {val_acc:.4f}, Test: {test_acc:.4f}, LR: {current_lr:.6f}")
                
                # Best model tracking
                if val_acc > best_run_val_acc:
                    best_run_val_acc = val_acc
                    best_run_test_acc = test_acc

                    if test_acc > global_best_test_acc:
                        global_best_test_acc = test_acc
                        torch.save(model.state_dict(), global_save_path)

    all_test_results.append(best_run_test_acc)
    print(f"Run {run+1} Finished. Best Test Acc for this run: {best_run_test_acc:.4f}")

# 4. Final Reporting
mean_acc = np.mean(all_test_results)
std_acc = np.std(all_test_results)

print("\n" + "="*30)
print(f"FINAL METRICS OVER {num_runs} RUNS")
print(f"Test Accuracy: {mean_acc:.4f} ± {std_acc:.4f}")
print("="*30)


>>> Starting Run 1/10 (Seed: 0)
Epoch 010: Val: 0.4559, Test: 0.4198, LR: 0.005000
Epoch 020: Val: 0.5525, Test: 0.5201, LR: 0.005000
Epoch 030: Val: 0.6378, Test: 0.6128, LR: 0.005000
Epoch 040: Val: 0.6780, Test: 0.6600, LR: 0.005000
Epoch 050: Val: 0.7124, Test: 0.6979, LR: 0.005000
Epoch 060: Val: 0.7113, Test: 0.6899, LR: 0.005000
Epoch 070: Val: 0.7240, Test: 0.7120, LR: 0.005000
Epoch 080: Val: 0.7242, Test: 0.7056, LR: 0.005000
Epoch 090: Val: 0.7158, Test: 0.7035, LR: 0.005000
Epoch 100: Val: 0.7345, Test: 0.7217, LR: 0.005000
Epoch 110: Val: 0.7360, Test: 0.7280, LR: 0.005000
Epoch 120: Val: 0.7379, Test: 0.7232, LR: 0.005000
Epoch 130: Val: 0.7045, Test: 0.6993, LR: 0.005000
Epoch 140: Val: 0.7408, Test: 0.7307, LR: 0.005000
Epoch 150: Val: 0.7349, Test: 0.7180, LR: 0.005000
Epoch 160: Val: 0.7435, Test: 0.7348, LR: 0.005000
Epoch 170: Val: 0.7443, Test: 0.7376, LR: 0.005000
Epoch 180: Val: 0.7247, Test: 0.6967, LR: 0.005000
Epoch 190: Val: 0.7292, Test: 0.7193, LR: 0.00500